# Volatility Forecasting — Vanilla RNN
**Pipeline:** Data → Dataset → Chronological Splits → Train → Evaluate

File layout:
```
project/
├── dataloader.py   # VolatilityDataset + prepare_data()
├── model.py        # VanillaRNN
├── train.py        # train_one_epoch / evaluate / run_training
├── main_analysis.ipynb  ← you are here
└── stock_data.pkl  # auto-generated cache
```

## 0. Imports & Device

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from dataloader import prepare_data, VolatilityDataset, TICKERS
from model import VanillaRNN
from train import run_training, evaluate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Load & Inspect Data

In [ ]:
returns, target_vol = prepare_data(force_download=False)

print('Returns shape:    ', returns.shape)    # (trading_days, num_stocks)
print('Target vol shape: ', target_vol.shape)
print('Date range:       ', returns.index[0], '→', returns.index[-1])

## 2. Chronological Train / Val / Test Splits

Per the spec:
- **Train:** 2000–2017
- **Val:** 2018, starting at the 21st trading day (buffer for last training target window)
- **Test:** 2019–2020, starting at the 21st trading day of 2019

In [ ]:
# --- Slice DataFrames by date ---
train_returns = returns.loc[:'2017-12-31']
train_target  = target_vol.loc[:'2017-12-31']

# Val: all of 2018; the Dataset's lookback handles the buffer naturally
# because the first valid sample in 2018 requires 60 days of prior returns,
# which are drawn from training history — no target leakage.
val_returns = returns.loc['2018-01-01':'2018-12-31']
val_target  = target_vol.loc['2018-01-01':'2018-12-31']

# Test: 2019–2020 (same reasoning)
test_returns = returns.loc['2019-01-01':'2020-12-31']
test_target  = target_vol.loc['2019-01-01':'2020-12-31']

print(f'Train days: {len(train_returns)}  |  '
      f'Val days: {len(val_returns)}  |  '
      f'Test days: {len(test_returns)}')

## 3. Build Datasets & DataLoaders

> **Lookback window** — experiment with `LOOKBACK = 10` (short) vs `60` (long) per the spec.

In [ ]:
LOOKBACK   = 60   # swap to 10 for the short-lookback experiment
BATCH_SIZE = 256

train_dataset = VolatilityDataset(train_returns, train_target, lookback=LOOKBACK)
val_dataset   = VolatilityDataset(val_returns,   val_target,   lookback=LOOKBACK)
test_dataset  = VolatilityDataset(test_returns,  test_target,  lookback=LOOKBACK)

# shuffle=True only for training; val/test keep chronological order
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train samples: {len(train_dataset):,}')
print(f'Val samples:   {len(val_dataset):,}')
print(f'Test samples:  {len(test_dataset):,}')

## 4. Instantiate Model & Optimizer

In [ ]:
NUM_STOCKS  = len(TICKERS)   # 99
EMBED_DIM   = 16
HIDDEN_SIZE = 64
DROPOUT     = 0.1

model = VanillaRNN(
    num_stocks=NUM_STOCKS,
    embed_dim=EMBED_DIM,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTrainable parameters: {total_params:,}')

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)

## 5. Train

In [ ]:
history = run_training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=DEVICE,
    num_epochs=30,
    patience=5,
    checkpoint_path=f'rnn_lookback{LOOKBACK}_best.pt',
    verbose=True,
)

## 6. Learning Curves

In [ ]:
epochs = range(1, len(history['train_mse']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_mse'], label='Train MSE')
axes[0].plot(epochs, history['val_mse'],   label='Val MSE')
axes[0].set_title('MSE (log-vol space)'); axes[0].legend()

axes[1].plot(epochs, history['val_mae'])
axes[1].set_title('Val MAE (original vol space)')

axes[2].plot(epochs, history['val_corr'])
axes[2].set_title('Val Pearson Correlation')

plt.tight_layout()
plt.savefig(f'learning_curves_lookback{LOOKBACK}.png', dpi=150)
plt.show()

## 7. Test Set Evaluation (full 2019–2020)

In [ ]:
test_metrics = evaluate(model, test_loader, DEVICE)

print('=== Test Results ===')
print(f'  MSE (log-vol):        {test_metrics["mse"]:.5f}')
print(f'  MAE (original vol):   {test_metrics["mae"]:.5f}')
print(f'  Pearson Correlation:  {test_metrics["correlation"]:.4f}')

## 8. Regime Analysis — 2019 vs 2020

2020 is the COVID crash: high-volatility regime. Comparing the two years
shows how the model handles regime changes.

In [ ]:
# Separate 2019 vs 2020 from the test set
returns_2019 = returns.loc['2019-01-01':'2019-12-31']
target_2019  = target_vol.loc['2019-01-01':'2019-12-31']
returns_2020 = returns.loc['2020-01-01':'2020-12-31']
target_2020  = target_vol.loc['2020-01-01':'2020-12-31']

ds_2019 = VolatilityDataset(returns_2019, target_2019, lookback=LOOKBACK)
ds_2020 = VolatilityDataset(returns_2020, target_2020, lookback=LOOKBACK)

loader_2019 = DataLoader(ds_2019, batch_size=BATCH_SIZE, shuffle=False)
loader_2020 = DataLoader(ds_2020, batch_size=BATCH_SIZE, shuffle=False)

m2019 = evaluate(model, loader_2019, DEVICE)
m2020 = evaluate(model, loader_2020, DEVICE)

print('\n--- 2019 (Low Volatility Regime) ---')
print(f'  MSE: {m2019["mse"]:.5f} | MAE: {m2019["mae"]:.5f} | Corr: {m2019["correlation"]:.4f}')
print('\n--- 2020 (COVID Crash — High Vol Regime) ---')
print(f'  MSE: {m2020["mse"]:.5f} | MAE: {m2020["mae"]:.5f} | Corr: {m2020["correlation"]:.4f}')

## 9. Prediction vs Realized Vol (sample stocks)

In [ ]:
# Plot pred vs true log-vol for the full test set (aggregated across stocks)
preds  = test_metrics['preds_log']
actual = test_metrics['targets_log']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(actual, preds, alpha=0.15, s=5)
lims = [min(actual.min(), preds.min()), max(actual.max(), preds.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1)
axes[0].set_xlabel('Realized log-vol'); axes[0].set_ylabel('Predicted log-vol')
axes[0].set_title(f'Pred vs True  (corr={test_metrics["correlation"]:.3f})')

# Time series — first 500 samples (mixed stocks, for illustration)
axes[1].plot(actual[:500], label='Realized', alpha=0.8)
axes[1].plot(preds[:500],  label='Predicted', alpha=0.8)
axes[1].set_title('Realized vs Predicted (first 500 test samples)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'pred_vs_true_lookback{LOOKBACK}.png', dpi=150)
plt.show()

## 10. Short vs Long Lookback Comparison

Re-run cells 3–7 with `LOOKBACK = 10`, save as a separate checkpoint,
then load both here for a side-by-side comparison.

In [ ]:
# Example comparison table (fill in after running both lookback experiments)
results = {
    'Lookback': [10, 60],
    'Test MSE': [None, test_metrics['mse']],     # fill in lookback-10 after re-run
    'Test MAE': [None, test_metrics['mae']],
    'Test Corr': [None, test_metrics['correlation']],
}
pd.DataFrame(results).set_index('Lookback')